<a href="https://colab.research.google.com/github/JohnnySolo/Data-Analysis-Project---Blockbuster-Movies/blob/main/preprocessing_notebook_4th_edition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Introduction & Preprocessing Methodology: The Greenlight Algorithm 🚦

## The 4 Pillars of Preprocessing
To prepare our raw data for the Greenlight Algorithm, we will adhere to a rigorous, production-grade preprocessing pipeline divided into four phases:

### I. Data Cleaning & Integrity (The Foundation)
Before any modeling occurs, we must ensure the dataset is structurally sound and logically possible.
* **Sanity Checks & Missing Values:** Financial models cannot tolerate placeholder data. For example, reported budgets of `$0` are physically impossible and will be strictly converted to `NaN` to prevent mathematical errors in ROI calculations. We will check data types, handle missing values, and filter out physically impossible records.
* **Single Source of Truth (SSOT):** When merging multiple datasets (e.g., TMDB, The Numbers), we will resolve overlapping columns (like `budget` vs. `production_budget`) by defining and documenting a clear SSOT based on data completeness and precision.

### II. Financial Standardization (Macroeconomics)
Time-series financial data contains inherent macroeconomic bias. A `$100M` budget in 1990 carries vastly different risk than a `$100M` budget in 2024.
* **Real vs. Nominal Dollars:** All financial metrics (Budget, Gross Revenue) will be adjusted to **Real 2024 Dollars** using historical Consumer Price Index (CPI) data. This ensures the algorithm evaluates risk on a level playing field across decades.

### III. Feature Engineering (Signals & NLP)
Raw data (names, text, dates) must be translated into mathematical "signals" based on the **R.I.C.E. Product Framework** (Reach, Impact, Confidence, Effort) to guide our analysis:
* **Confidence Signals (Historical Authority):** Calculating pre-release "Win Rates" for Directors, Writers, and Stars to act as a proxy for execution risk.
* **Reach Signals (Natural Language Processing):** Utilizing NLP (`CountVectorizer`) on plot overviews to mathematically extract and flag high-value semantic themes (e.g., "Family", "War") rather than relying on brittle manual keyword searches.
* **Timing Signals (Release Strategy):** Extracting seasonality (Blockbuster vs. Dump Months) and weekday data to proxy strategic market placement.

### IV. Target Formulation & Leakage Prevention (The North Star)
The golden rule of predictive modeling is preventing the model from "cheating" by seeing the future.
* **Financial Target:** Our primary target variable is a strict financial classifier (`is_profitable`), derived from real revenue and real budget.
* **Leakage Prevention:** Post-release metrics like `popularity` or `vote_average` will *not* be used as predictive features. Instead, historical proxy scores will be used as pre-release risk multipliers, ensuring an absolute boundary between pre-launch inputs and post-launch outcomes.

---

## Project Data Sources

**1. Movie Data Analysis Dataset**  
- Details about 7,668 movies, including:
  - Titles, ratings, genres, release years
  - IMDb scores, votes
  - Directors, writers, main stars
  - Production countries, budgets, gross earnings
  - Production companies, runtimes  
- **Source**: [GitHub Repository](https://github.com/1tannu5/Movie-Data-Analysis?utm_source=chatgpt.com)

---

**2. Global Movie Franchise Revenue and Budget Data**  
- Comprehensive data on movie franchises worldwide between 2000–2020:
  - Lifetime gross, budget, rating
  - Runtime, release date, vote count/average  
- **Source**: [Kaggle Dataset](https://www.kaggle.com/datasets/thedevastator/global-movie-franchise-revenue-and-budget-data?utm_source=chatgpt.com)

---

**3. TMDB 5000 Movies Dataset**  
- Information on over 5,000 movies:
  - Budget, cast, director
  - Keywords, runtime, genres
  - Production companies, release dates  
- **Source**: [Hugging Face Dataset](https://huggingface.co/datasets/AiresPucrs/tmdb-5000-movies/blob/main/README.md?utm_source=chatgpt.com)

---

**4. Complete Movie Metadata Dataset**  
- Data on over 722,000 movies, including:
  - ID, title, genres, budget, revenue  
- Suitable for analyzing trends in movie popularity, production companies, budgets, and revenues.  
- **Source**: [Gigasheet Dataset](https://www.gigasheet.com/sample-data/movies-daily-update-dataset?utm_source=chatgpt.com)

---

**5. Movie Revenue Analysis Dataset**  
- Approx. 1,800 movies released between 1915 and 2020:
  - Domestic and worldwide gross revenues
  - Production budgets, release dates  
- **Source**: [GitHub Repository](https://github.com/ntdoris/movie-revenue-analysis?utm_source=chatgpt.com)

# Extraction

In [ ]:
import pandas as pd
from IPython.display import display

def quick_column_summary(df, table_name):
    """
    Generates a diagnostic summary of a DataFrame's columns,
    including data types, missing value counts, and missing percentages.
    """
    print(f"\n📋 Column Summary for `{table_name}`\n")

    summary = pd.DataFrame({
        'Data Type': df.dtypes,
        'NA Count': df.isna().sum(),
        '% Missing': (df.isna().mean() * 100).round(2)
    }).reset_index().rename(columns={'index': 'Column'})

    summary = summary.sort_values(by='% Missing', ascending=False).reset_index(drop=True)

    display(summary)

## 1st Dataset: Movie Franchises



In [ ]:
# 1. Movie Data Analysis Dataset (Base Table)
!wget https://raw.githubusercontent.com/JohnnySolo/Data-Analysis-Project---Blockbuster-Movies/main/movie.csv -O movie.csv

movie_franchises = pd.read_csv("movie.csv")

# Rename the IMDB score column (Kept for potential EDA, but no longer our target)
movie_franchises = movie_franchises.rename(columns={"score": "imdb_score"})

--2026-03-19 17:59:30--  https://raw.githubusercontent.com/JohnnySolo/Data-Analysis-Project---Blockbuster-Movies/main/movie.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1294548 (1.2M) [text/plain]
Saving to: ‘movie.csv’

movie.csv           100%[===================>]   1.23M  --.-KB/s    in 0.01s   

2026-03-19 17:59:30 (95.8 MB/s) - ‘movie.csv’ saved [1294548/1294548]



### First check

In [ ]:
# Display the first few rows
movie_franchises.head()

,name,rating,genre,year,released,imdb_score,votes,director,writer,star,country,budget,gross,company,runtime
0,The Shining,R,Drama,1980,"June 13, 1980 (United States)",8.4,927000.0,Stanley Kubrick,Stephen King,Jack Nicholson,United Kingdom,19000000.0,46998772.0,Warner Bros.,146.0
1,The Blue Lagoon,R,Adventure,1980,"July 2, 1980 (United States)",5.8,65000.0,Randal Kleiser,Henry De Vere Stacpoole,Brooke Shields,United States,4500000.0,58853106.0,Columbia Pictures,104.0
2,Star Wars: Episode V - The Empire Strikes Back,PG,Action,1980,"June 20, 1980 (United States)",8.7,1200000.0,Irvin Kershner,Leigh Brackett,Mark Hamill,United States,18000000.0,538375067.0,Lucasfilm,124.0
3,Airplane!,PG,Comedy,1980,"July 2, 1980 (United States)",7.7,221000.0,Jim Abrahams,Jim Abrahams,Robert Hays,United States,3500000.0,83453539.0,Paramount Pictures,88.0
4,Caddyshack,R,Comedy,1980,"July 25, 1980 (United States)",7.3,108000.0,Harold Ramis,Brian Doyle-Murray,Chevy Chase,United States,6000000.0,39846344.0,Orion Pictures,98.0


In [ ]:
# Display the dataset shape
movie_franchises.shape

(7668, 15)

In [ ]:
# --- FINANCIAL NORTH STAR ADJUSTMENTS ---

import pandas as pd
import numpy as np

# Fix 1: Treat $0 as Missing Data (Data Integrity)
movie_franchises['budget'] = movie_franchises['budget'].replace(0, np.nan)
movie_franchises['gross'] = movie_franchises['gross'].replace(0, np.nan)

In [ ]:
# Check for data types and NA's
quick_column_summary(movie_franchises, 'movie_franchises')


📋 Column Summary for `movie_franchises`



,Column,Data Type,NA Count,% Missing
0,budget,float64,2171,28.31
1,gross,float64,189,2.46
2,rating,object,77,1.00
3,company,object,17,0.22
4,runtime,float64,4,0.05
5,imdb_score,float64,3,0.04
6,votes,float64,3,0.04
7,country,object,3,0.04
8,writer,object,3,0.04
9,released,object,2,0.03


In [ ]:
# Fix 2: Omit observations with NA's in FINANCIAL target variables ONLY
movie_franchises = movie_franchises[
    movie_franchises['budget'].notna() &
    movie_franchises['gross'].notna()
].copy()

In [ ]:
# Display the dataset shape
movie_franchises.shape

(5436, 15)

## 2nd Dataset: additional Movie Franchises

In [ ]:
# 2. Global Movie Franchise Revenue and Budget Data

!wget https://raw.githubusercontent.com/JohnnySolo/Data-Analysis-Project---Blockbuster-Movies/main/MovieFranchises.csv -O MovieFranchises.csv
import pandas as pd
data2 = pd.read_csv("MovieFranchises.csv") # Save in a different name due to similar name to the 1st dataset

--2026-03-19 17:59:58--  https://raw.githubusercontent.com/JohnnySolo/Data-Analysis-Project---Blockbuster-Movies/main/MovieFranchises.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26322 (26K) [text/plain]
Saving to: ‘MovieFranchises.csv’

MovieFranchises.csv 100%[===================>]  25.71K  --.-KB/s    in 0s      

2026-03-19 17:59:59 (95.8 MB/s) - ‘MovieFranchises.csv’ saved [26322/26322]



### First check

In [ ]:
# Display the first few rows
data2.head()

,index,MovieID,Title,Lifetime Gross,Year,Studio,Rating,Runtime,Budget,ReleaseDate,VoteAvg,VoteCount,FranchiseID
0,0,1001,Star Wars: Episode IV - A New Hope,775398007,1977,Lucasfilm,PG,121.0,11000000.0,05-25-77,4.09,96233.0,101.0
1,1,1002,Star Wars: Episode V - The Empire Strikes Back,538375067,1980,Lucasfilm,PG,124.0,18000000.0,06-20-80,4.12,79231.0,101.0
2,2,1003,Star Wars: Episode VI - Return of the Jedi,475106177,1983,Lucasfilm,PG,135.0,32500000.0,05-25-83,3.98,76082.0,101.0
3,3,1004,Jurassic Park,1109802321,1993,Universal Pictures,PG-13,127.0,63000000.0,06-11-93,3.69,82700.0,102.0
4,4,1005,The Lost World: Jurassic Park,618638999,1997,Universal Pictures,PG-13,129.0,73000000.0,05-23-97,3.01,19721.0,102.0


In [ ]:
# Display the dataset shape
data2.shape

(605, 13)

In [ ]:
# --- FINANCIAL NORTH STAR ADJUSTMENTS ---

import pandas as pd
import numpy as np

# Fix 1: Treat $0 as Missing Data (Data Integrity)
data2['Budget'] = data2['Budget'].replace(0, np.nan)
data2['Lifetime Gross'] = data2['Lifetime Gross'].replace(0, np.nan)

In [ ]:
# Check for data types and NA's
quick_column_summary(data2, 'data2')


📋 Column Summary for `data2`



,Column,Data Type,NA Count,% Missing
9,ReleaseDate,object,545,90.08
8,Budget,float64,545,90.08
7,Runtime,float64,545,90.08
6,Rating,object,545,90.08
5,Studio,object,545,90.08
11,VoteCount,float64,545,90.08
10,VoteAvg,float64,545,90.08
12,FranchiseID,float64,545,90.08
4,Year,object,539,89.09
3,Lifetime Gross,object,0,0.00


In [ ]:
# Keep only the useful parts of data2
data2 = data2[['MovieID', 'Title', 'Lifetime Gross']].copy()

## 3rd Dataset: TMDB data

In [ ]:
# If the 3rd dataset have error contains "LocalFileSystem is not supported" then use the code:
# pip install -U datasets

In [ ]:
# 3. TMDB 5000 Movies Dataset

!pip install datasets

from datasets import load_dataset
import pandas as pd

# Load the TMDB dataset from Hugging Face
dataset = load_dataset("AiresPucrs/tmdb-5000-movies", split="train")
tmdb_data = pd.DataFrame(dataset)

# Save the DataFrame to a CSV file
tmdb_data.to_csv("tmdb_movies.csv", index=False)

# Confirm the file exists in the current directory
import os
os.listdir()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-6db04ab1c75d68(…):   0%|          | 0.00/13.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4803 [00:00<?, ? examples/s]

['.config',
 'movie.csv',
 'tmdb_movies.csv',
 'MovieFranchises.csv',
 'sample_data']

### First Check

In [ ]:
# Display the first few rows
tmdb_data.head()

,id,budget,genres,homepage,keywords,original_language,original_title,overview,popularity,production_companies,...,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew
0,5,4000000,"[{""id"": 80, ""name"": ""Crime""}, {""id"": 35, ""name...",None,"[{""id"": 612, ""name"": ""hotel""}, {""id"": 613, ""na...",en,Four Rooms,It's Ted the Bellhop's first night on the job....,22.876230,"[{""name"": ""Miramax Films"", ""id"": 14}, {""name"":...",...,4300000,98.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Twelve outrageous guests. Four scandalous requ...,Four Rooms,6.5,530,"[{""cast_id"": 42, ""character"": ""Ted the Bellhop...","[{""credit_id"": ""52fe420dc3a36847f800012d"", ""de..."
1,11,11000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 28, ""...",http://www.starwars.com/films/star-wars-episod...,"[{""id"": 803, ""name"": ""android""}, {""id"": 4270, ...",en,Star Wars,Princess Leia is captured and held hostage by ...,126.393695,"[{""name"": ""Lucasfilm"", ""id"": 1}, {""name"": ""Twe...",...,775398007,121.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"A long time ago in a galaxy far, far away...",Star Wars,8.1,6624,"[{""cast_id"": 3, ""character"": ""Luke Skywalker"",...","[{""credit_id"": ""52fe420dc3a36847f8000437"", ""de..."
2,12,94000000,"[{""id"": 16, ""name"": ""Animation""}, {""id"": 10751...",http://movies.disney.com/finding-nemo,"[{""id"": 494, ""name"": ""father son relationship""...",en,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...",85.688789,"[{""name"": ""Pixar Animation Studios"", ""id"": 3}]",...,940335536,100.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"There are 3.7 trillion fish in the ocean, they...",Finding Nemo,7.6,6122,"[{""cast_id"": 8, ""character"": ""Marlin (voice)"",...","[{""credit_id"": ""52fe420ec3a36847f80006b1"", ""de..."
3,13,55000000,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...",None,"[{""id"": 422, ""name"": ""vietnam veteran""}, {""id""...",en,Forrest Gump,A man with a low IQ has accomplished great thi...,138.133331,"[{""name"": ""Paramount Pictures"", ""id"": 4}]",...,677945399,142.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"The world will never be the same, once you've ...",Forrest Gump,8.2,7927,"[{""cast_id"": 7, ""character"": ""Forrest Gump"", ""...","[{""credit_id"": ""52fe420ec3a36847f800076b"", ""de..."
4,14,15000000,"[{""id"": 18, ""name"": ""Drama""}]",http://www.dreamworks.com/ab/,"[{""id"": 255, ""name"": ""male nudity""}, {""id"": 29...",en,American Beauty,"Lester Burnham, a depressed suburban father in...",80.878605,"[{""name"": ""DreamWorks SKG"", ""id"": 27}, {""name""...",...,356296601,122.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Look closer.,American Beauty,7.9,3313,"[{""cast_id"": 6, ""character"": ""Lester Burnham"",...","[{""credit_id"": ""52fe420ec3a36847f8000809"", ""de..."


In [ ]:
# Display the dataset shape
tmdb_data.shape

(4803, 22)

In [ ]:
# --- FINANCIAL NORTH STAR ADJUSTMENTS ---

import pandas as pd
import numpy as np

# Fix 1: Treat $0 as Missing Data (Data Integrity)
tmdb_data['budget'] = tmdb_data['budget'].replace(0, np.nan)
tmdb_data['revenue'] = tmdb_data['revenue'].replace(0, np.nan)

In [ ]:
# Check for data types and NA's
quick_column_summary(tmdb_data, 'tmdb_data')


📋 Column Summary for `tmdb_data`



,Column,Data Type,NA Count,% Missing
3,homepage,object,3091,64.36
12,revenue,float64,1427,29.71
1,budget,float64,1037,21.59
16,tagline,object,844,17.57
7,overview,object,3,0.06
13,runtime,float64,2,0.04
11,release_date,object,1,0.02
4,keywords,object,0,0.00
2,genres,object,0,0.00
0,id,int64,0,0.00


In [ ]:
# Fix 2: Omit observations with NA's in FINANCIAL target variables ONLY
tmdb_data = tmdb_data[
    tmdb_data['budget'].notna() &
    tmdb_data['revenue'].notna()
].copy()

## 4th Dataset: Meta-Analysis Data

In [ ]:
# 4. Complete Movie Metadata Dataset

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
file_path = '/content/drive/My Drive/Projects/Blockbuster Movies/movies.csv'  # Adjust path as needed
meta_data = pd.read_csv(file_path)

# Save the DataFrame to a CSV file
meta_data.to_csv("movies.csv", index=False)

# Confirm the file exists in the current directory
import os
os.listdir()

Mounted at /content/drive


['.config',
 'movie.csv',
 'tmdb_movies.csv',
 'drive',
 'movies.csv',
 'MovieFranchises.csv',
 'sample_data']

### First Check

In [ ]:
# Display the first few rows
meta_data.head()

,id,title,genres,original_language,overview,popularity,production_companies,release_date,budget,revenue,runtime,status,tagline,vote_average,vote_count,credits,keywords,poster_path,backdrop_path,recommendations
0,615656,Meg 2: The Trench,Action-Science Fiction-Horror,en,An exploratory dive into the deepest depths of...,8763.998,Apelles Entertainment-Warner Bros. Pictures-di...,2023-08-02,129000000.0,3.520565e+08,116.0,Released,Back for seconds.,7.079,1365.0,Jason Statham-Wu Jing-Shuya Sophia Cai-Sergio ...,based on novel or book-sequel-kaiju,/4m1Au3YkjqsxF8iwQy0fPYSxE0h.jpg,/qlxy8yo5bcgUw2KAmmojUKp4rHd.jpg,1006462-298618-569094-1061181-346698-1076487-6...
1,758323,The Pope's Exorcist,Horror-Mystery-Thriller,en,Father Gabriele Amorth Chief Exorcist of the V...,5953.227,Screen Gems-2.0 Entertainment-Jesus & Mary-Wor...,2023-04-05,18000000.0,6.567582e+07,103.0,Released,Inspired by the actual files of Father Gabriel...,7.433,545.0,Russell Crowe-Daniel Zovatto-Alex Essoe-Franco...,spain-rome italy-vatican-pope-pig-possession-c...,/9JBEPLTPSm0d1mbEcLxULjJq9Eh.jpg,/hiHGRbyTcbZoLsYYkO4QiCLYe34.jpg,713704-296271-502356-1076605-1084225-1008005-9...
2,533535,Deadpool & Wolverine,Action-Comedy-Science Fiction,en,A listless Wade Wilson toils away in civilian ...,5410.496,Marvel Studios-Maximum Effort-21 Laps Entertai...,2024-07-24,200000000.0,1.326387e+09,128.0,Released,Come together.,7.765,3749.0,Ryan Reynolds-Hugh Jackman-Emma Corrin-Matthew...,hero-superhero-anti hero-mutant-breaking the f...,/8cdWjvZQUExUUTzyp4t6EDMubfO.jpg,/dvBCdCohwWbsP5qAaglOXagDMtk.jpg,573435-519182-957452-1022789-945961-718821-103...
3,667538,Transformers: Rise of the Beasts,Action-Adventure-Science Fiction,en,When a new threat capable of destroying the en...,5409.104,Skydance-Paramount-di Bonaventura Pictures-Bay...,2023-06-06,200000000.0,4.070455e+08,127.0,Released,Unite or fall.,7.340,1007.0,Anthony Ramos-Dominique Fishback-Luna Lauren V...,peru-alien-end of the world-based on cartoon-b...,/gPbM0MK8CP8A174rmUwGsADNYKD.jpg,/woJbg7ZqidhpvqFGGMRhWQNoxwa.jpg,496450-569094-298618-385687-877100-598331-4628...
4,693134,Dune: Part Two,Science Fiction-Adventure,en,Follow the mythic journey of Paul Atreides as ...,4742.163,Legendary Pictures,2024-02-27,190000000.0,6.838137e+08,167.0,Released,Long live the fighters.,8.300,2770.0,Timothée Chalamet-Zendaya-Rebecca Ferguson-Jav...,epic-based on novel or book-fight-sandstorm-sa...,/czembW0Rk1Ke7lCJGahbOhdCuhV.jpg,/xOMo8BRK7PfcJv9JCnx7s5hj0PX.jpg,438631-763215-792307-1011985-467244-634492-359...


In [ ]:
# Display the dataset shape
meta_data.shape

(722317, 20)

In [ ]:
# Count rows where budget is 0
zero_budget_count = (meta_data['budget'] == 0).sum()
print(f"Number of rows with budget = 0: {zero_budget_count}")

# Count rows where revenue is 0
zero_revenue_count = (meta_data['revenue'] == 0).sum()
print(f"Number of rows with revenue = 0: {zero_revenue_count}")

Number of rows with budget = 0: 685547
Number of rows with revenue = 0: 705113


In [ ]:
# --- FINANCIAL NORTH STAR ADJUSTMENTS ---

import pandas as pd
import numpy as np

# Fix 1: Treat $0 as Missing Data (Data Integrity)
meta_data['budget'] = meta_data['budget'].replace(0, np.nan)
meta_data['revenue'] = meta_data['revenue'].replace(0, np.nan)

In [ ]:
# Check for data types and NA's
quick_column_summary(meta_data, 'meta_data')


📋 Column Summary for `meta_data`



,Column,Data Type,NA Count,% Missing
9,revenue,float64,705113,97.62
19,recommendations,object,686301,95.01
8,budget,float64,685547,94.91
12,tagline,object,613841,84.98
16,keywords,object,511678,70.84
18,backdrop_path,object,499106,69.10
6,production_companies,object,384926,53.29
15,credits,object,224714,31.11
2,genres,object,210317,29.12
17,poster_path,object,184493,25.54


In [ ]:
# Fix 2: Omit observations with NA's in FINANCIAL target variables ONLY
meta_data = meta_data[
    meta_data['budget'].notna() &
    meta_data['revenue'].notna()
].copy()

## 5th Dataset: Revenues Data

In [ ]:
# 5. Movie Revenue Analysis Dataset

!wget https://raw.githubusercontent.com/JohnnySolo/Data-Analysis-Project---Blockbuster-Movies/main/final_dataset.csv -O final_dataset.csv
import pandas as pd
financial_data = pd.read_csv("final_dataset.csv")

--2026-03-19 18:02:44--  https://raw.githubusercontent.com/JohnnySolo/Data-Analysis-Project---Blockbuster-Movies/main/final_dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 456039 (445K) [text/plain]
Saving to: ‘final_dataset.csv’

final_dataset.csv   100%[===================>] 445.35K  --.-KB/s    in 0.003s  

2026-03-19 18:02:44 (146 MB/s) - ‘final_dataset.csv’ saved [456039/456039]



### First Check

In [ ]:
# Display the first few rows
financial_data.head()

,Unnamed: 0,movie,year,production_budget,domestic_gross,foreign_gross,worldwide_gross,month,profit,profit_margin,...,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western
0,0,Avatar,2009,425000000,760507625,2015837654,2776345279,12,2351345279,0.846921,...,0,0,0,0,0,1,0,0,0,0
1,1,Pirates of the Caribbean: On Stranger Tides,2011,410600000,241063875,804600000,1045663875,5,635063875,0.607331,...,0,0,0,0,0,0,0,0,0,0
2,2,Avengers: Age of Ultron,2015,330600000,459005868,944008095,1403013963,5,1072413963,0.764364,...,0,0,0,0,0,1,0,0,0,0
3,3,Avengers: Infinity War,2018,300000000,678815482,1369318718,2048134200,4,1748134200,0.853525,...,0,0,0,0,0,0,0,0,0,0
4,4,Justice League,2017,300000000,229024295,426920914,655945209,11,355945209,0.542645,...,0,0,0,0,0,1,0,0,0,0


In [ ]:
# Display the dataset shape
financial_data.shape

(1759, 39)

In [ ]:
# --- FINANCIAL NORTH STAR ADJUSTMENTS ---

import pandas as pd
import numpy as np

# Fix 1: Treat $0 as Missing Data (Data Integrity)
financial_data['production_budget'] = financial_data['production_budget'].replace(0, np.nan)
financial_data['worldwide_gross'] = financial_data['worldwide_gross'].replace(0, np.nan)
financial_data['profit'] = financial_data['profit'].replace(0, np.nan)

In [ ]:
# Check for data types and NA's
quick_column_summary(financial_data, 'financial_data')


📋 Column Summary for `financial_data`



,Column,Data Type,NA Count,% Missing
6,worldwide_gross,float64,101,5.74
19,genres,object,8,0.45
1,movie,object,0,0.00
2,year,int64,0,0.00
3,production_budget,int64,0,0.00
4,domestic_gross,int64,0,0.00
0,Unnamed: 0,int64,0,0.00
5,foreign_gross,int64,0,0.00
7,month,int64,0,0.00
9,profit_margin,float64,0,0.00


In [ ]:
# Fix 2: Omit observations with NA's in FINANCIAL target variables ONLY
financial_data = financial_data[
    financial_data['production_budget'].notna() &
    financial_data['worldwide_gross'].notna() &
        financial_data['profit'].notna()
].copy()

## Data Acquisition & Dictionary Summaries

*The following tables outline the raw structure and relevance of the 5 primary datasets ingested into the pipeline.*

### 📥 Table 1: `movie_franchises`
The foundational table containing baseline financial and personnel data.

| Column | Description | Relevance |
| :--- | :--- | :--- |
| `name` | Movie name (key) | ✅ Unique ID across datasets |
| `budget`, `gross` | Financials in dollars | 📌 Required for ROI (target) |
| `score`, `votes` | IMDB rating and volume | 🧪 Post-release audience signals |
| `genre`, `rating`, `year`, `released` | Movie metadata | 📊 Baseline features |
| `director`, `writer`, `star`, `company` | Key personnel / studio | 📊 Personnel features |
| `runtime` | Duration in minutes | 📊 Feature (scale/scope proxy) |
| `country` | Country of production | 📊 Feature for cultural origin |

---

### 📥 Table 2: `data2`
A supplementary table primarily utilized for gap-filling missing financial data.

| `movie_franchises` | `data2` | Action |
| :--- | :--- | :--- |
| `name` | `Title` | Normalize to `movie_id` for matching |
| `budget` | `Budget` | Compare and retain best version (SSOT) |
| `gross` | `Lifetime Gross` | Compare with `gross` (SSOT) |

---

### 📥 Table 3: `tmdb_data`
The richest structured dataset. It includes JSON-nested data and contributes heavily to thematic and reach-based features.

| Column | Description | Relevance |
| :--- | :--- | :--- |
| `title` | Movie name | ✅ Used to create `movie_id` |
| `vote_average`, `vote_count` | Audience metrics | 🧪 Post-release audience signals |
| `budget`, `revenue` | Financial data | 📌 Required for ROI |
| `popularity` | TMDB popularity score | 📊 Social visibility signal |
| `release_date` | Date released | 📊 Source for release strategy features |
| `genres`, `keywords` | Thematic lists (JSON) | 🧠 To parse for demographic analysis |
| `overview`, `tagline` | Textual summaries | 🧠 Source text for NLP thematic modeling |
| `original_language` | Language code (e.g., 'en') | 📊 Cultural/demographic indicator |

---

### 📥 Table 4: `meta_data`
A highly overlapping, complementary version of `tmdb_data` containing recent and upcoming titles.

| Column | Description | Relevance |
| :--- | :--- | :--- |
| `title` | Movie title | ✅ Used to create `movie_id` |
| `vote_average`, `vote_count` | Audience metrics | 🧪 Post-release audience signals |
| `budget`, `revenue` | Financial data | 📌 Used for ROI |
| `runtime`, `release_date` | Timing & length | 📊 Influences cost & strategy |
| `genres`, `keywords`, `overview` | Text / tags | 🧠 Feature-rich text for parsing |

---

### 📥 Table 5: `financial_data`
A highly focused table containing pre-calculated financial metrics and cleanly encoded categorical data.

| Column | Description | Relevance |
| :--- | :--- | :--- |
| `movie` | Movie name | ✅ Used to create `movie_id` |
| `production_budget`, `worldwide_gross` | Raw financial inputs | ✅ SSOT Priority |
| `profit`, `roi` | Pre-calculated finance metrics | ✅ Verification points |
| `Action`, `Drama`, etc. | Binary genre flags | ✅ Pre-encoded demographic features |

---

# Transform

# Phase I: Data Consolidation & The SSOT Strategy

Merging five distinct datasets presents a significant data engineering challenge: managing overlapping columns (e.g., multiple `budget` columns) and inconsistent coverage. To resolve this, we execute a **Single Source of Truth (SSOT)** merging strategy designed to maximize our pool of financially valid movies without introducing duplicates or corrupted data.

### 1. Normalizing Identifiers (The Join Key)
To prepare for the merge, we create a primary key called `movie_id` across all datasets.
* **Action:** We extract the movie title, convert it to lowercase, and strip leading/trailing whitespace.
* **Safety Check:** We drop missing titles and duplicate `movie_id`s within each individual dataset *before* merging to prevent an outer join "explosion" (e.g., accidentally creating 50 rows for movies with common names).

### 2. The Merging Strategy: Outer Join
Instead of a restrictive Left Join, we utilize an **Outer Join** sequentially across all datasets.
* **The "Why":** A Left Join would artificially cap our dataset size to the ~5,400 rows of the base table. By outer joining, we capture every unique movie across all sources, evaluating a massive potential universe of over 700,000 films before applying our strict financial filters.

### 3. Establishing the Single Source of Truth (SSOT)
After the outer join, we are left with multiple conflicting columns for the same metric (e.g., `budget_base`, `budget_tmdb`).
* **The Coalesce Technique:** We use Pandas' `combine_first()` to establish an SSOT. We prioritize the most reliable data source (e.g., `production_budget`). If that value is missing (`NaN`), the algorithm automatically cascades down to the next available source (`budget_base`, then `budget_tmdb`, etc.) until a valid number is found.
* **Result:** We formulate definitive master columns (e.g., `ultimate_budget`, `ultimate_revenue`, `ultimate_year`).

### 4. The Financial North Star (The Great Purge)
Because our algorithm's strict objective is predicting financial ROI, any row missing an `ultimate_budget` or `ultimate_revenue` is structurally useless.
* **Action:** We drop any movie that lacks complete financial data across all sources. This aggressively filters the massive outer-joined dataset down to only the absolute highest-quality, financially verified records.

### 1. Primary Key Normalization

In [ ]:
# --- PHASE I (PART 1): DATA NORMALIZATION & UNIVERSE SIZING ---
import pandas as pd

def normalize_primary_key(df, title_col):
    """
    Creates a standardized primary key ('movie_id') from movie titles
    to enable accurate cross-dataset joining. Handles trailing spaces and casing.
    """
    df_clean = df.copy()

    # Pre-emptively drop missing titles to prevent "nan" ghost strings
    df_clean = df_clean.dropna(subset=[title_col])

    # Standardize text: convert to string, remove edge spaces, lowercase
    df_clean['movie_id'] = df_clean[title_col].astype(str).str.strip().str.lower()

    # Deduplicate to ensure 1-to-1 mapping
    return df_clean.drop_duplicates(subset=['movie_id']).reset_index(drop=True)


# 1. Normalize all datasets
movie_franchises = normalize_primary_key(movie_franchises, 'name')
data2 = normalize_primary_key(data2, 'Title')
tmdb_data = normalize_primary_key(tmdb_data, 'title')
meta_data = normalize_primary_key(meta_data, 'title')
financial_data = normalize_primary_key(financial_data, 'movie')

# 2. Define the base financial universe
base_movie_ids = set(movie_franchises['movie_id']).union(set(financial_data['movie_id']))
print(f"Base financial universe size: {len(base_movie_ids):,} movies\n")

# 3. Filter external datasets for strict financial validity (Budget > 0, Revenue > 0)
valid_meta = meta_data[(meta_data['budget'] > 0) & (meta_data['revenue'] > 0)]
valid_tmdb = tmdb_data[(tmdb_data['budget'] > 0) & (tmdb_data['revenue'] > 0)]

# 4. Calculate Net New additions
new_from_meta = set(valid_meta['movie_id']) - base_movie_ids
new_from_tmdb = set(valid_tmdb['movie_id']) - base_movie_ids - new_from_meta

print(f"Financially valid movies in meta_data: {len(valid_meta):,}")
print(f" 👉 Net New additions: {len(new_from_meta):,}\n")

print(f"Financially valid movies in tmdb_data: {len(valid_tmdb):,}")
print(f" 👉 Net New additions: {len(new_from_tmdb):,}\n")

total_universe = len(base_movie_ids) + len(new_from_meta) + len(new_from_tmdb)
print(f"🚀 Maximum Viable Dataset Size: {total_universe:,} movies")

Base financial universe size: 5,705 movies

Financially valid movies in meta_data: 10,613
 👉 Net New additions: 5,923

Financially valid movies in tmdb_data: 3,228
 👉 Net New additions: 39

🚀 Maximum Viable Dataset Size: 11,667 movies


### 2. The Merging Strategy: Outer Join

In [ ]:
# --- PHASE I (Part 2): Merging & Purging Data ---
import pandas as pd
import numpy as np

# 1. Pre-Merge Zero Purge
# Replace zeroes with NaNs in external datasets to prevent false zeros from skewing calculations
tmdb_data['budget'] = tmdb_data['budget'].replace(0, np.nan)
tmdb_data['revenue'] = tmdb_data['revenue'].replace(0, np.nan)
meta_data['budget'] = meta_data['budget'].replace(0, np.nan)
meta_data['revenue'] = meta_data['revenue'].replace(0, np.nan)
data2['Lifetime Gross'] = data2['Lifetime Gross'].replace(0, np.nan)

# 2. The Outer Join Sequence
# Successively merge all datasets using the normalized 'movie_id' as the primary key.
# Outer joins are used to preserve all possible records prior to the strict financial filter.
enriched = pd.merge(movie_franchises, financial_data, on='movie_id', how='outer', suffixes=('_base', '_fin'))
enriched = pd.merge(enriched, tmdb_data, on='movie_id', how='outer', suffixes=('', '_tmdb'))
enriched = pd.merge(enriched, meta_data, on='movie_id', how='outer', suffixes=('', '_meta'))
enriched = pd.merge(enriched, data2, on='movie_id', how='outer')

print(f"Total records consolidated after outer joins: {len(enriched):,}")

Total records consolidated after outer joins: 11,771


### 3. Establishing the Single Source of Truth (SSOT)

In [ ]:
# --- PHASE I (Part 3): ESTABLISHING THE FINANCIAL SINGLE SOURCE OF TRUTH (SSOT) ---

# 1. Safely extract all possible Budget and Revenue columns
# Using .get() ensures the code won't crash if a column name variation is missing
b_prod = enriched.get('production_budget', pd.Series(np.nan, index=enriched.index))
b_base = enriched.get('budget', pd.Series(np.nan, index=enriched.index))
b_tmdb = enriched.get('budget_tmdb', pd.Series(np.nan, index=enriched.index))
b_meta = enriched.get('budget_meta', pd.Series(np.nan, index=enriched.index))

r_world = enriched.get('worldwide_gross', pd.Series(np.nan, index=enriched.index))
r_base = enriched.get('gross', pd.Series(np.nan, index=enriched.index))
r_tmdb = enriched.get('revenue', pd.Series(np.nan, index=enriched.index))
r_meta = enriched.get('revenue_meta', pd.Series(np.nan, index=enriched.index))
r_data2 = enriched.get('Lifetime Gross', pd.Series(np.nan, index=enriched.index))

# 2. Execute Coalesce Strategy (Waterfall Prioritization)
# Combine_first() fills NaNs in the calling Series with values from the passed Series.
# Priority Order: Production/Worldwide -> Base Financials -> TMDB -> Meta -> Data2
enriched['ultimate_budget'] = b_prod.combine_first(b_base).combine_first(b_tmdb).combine_first(b_meta)

enriched['ultimate_revenue'] = (
    r_world.combine_first(r_base)
           .combine_first(r_tmdb)
           .combine_first(r_meta)
           .combine_first(r_data2)
)

print(f"Financial SSOT established. 'ultimate_budget' and 'ultimate_revenue' created.")

Financial SSOT established. 'ultimate_budget' and 'ultimate_revenue' created.


### 4. The Financial North Star (The Great Purge)

In [ ]:
# --- PHASE I (PART 4): THE FINANCIAL PURGE & CLEANUP ---

# 1. The Great Purge: Enforce the Financial North Star
# Drop any movie that failed to establish both a budget and a revenue in our SSOT
enriched_clean = enriched.dropna(subset=['ultimate_budget', 'ultimate_revenue']).copy()

# 2. Memory Cleanup: Remove redundant raw financial columns
raw_financial_cols = [
    'production_budget', 'budget', 'budget_tmdb', 'budget_meta',
    'worldwide_gross', 'gross', 'revenue', 'revenue_meta', 'Lifetime Gross'
]

# Safely drop columns only if they exist in the dataframe to free up RAM
cols_to_drop = [c for c in raw_financial_cols if c in enriched_clean.columns]
enriched_clean = enriched_clean.drop(columns=cols_to_drop)

print(f"🔥 Phase I Complete: {len(enriched_clean):,} financially verified movies ready for Phase II.")

🔥 Phase I Complete: 11,668 financially verified movies ready for Phase II.


In [ ]:
# Check the shape
enriched_clean.shape

(11668, 93)

# Phase II: Financial Standardization & Macroeconomics

A predictive financial model must evaluate risk on a level playing field. Comparing the `$11M` budget of *Star Wars* (1977) directly to the `$350M` budget of *Avengers: Endgame* (2019) introduces massive macroeconomic bias due to decades of inflation.

To solve this, Phase II applies strict temporal and financial standardization:

1. **Temporal SSOT (The Release Year):** Before calculating inflation, we must definitively know when the money was spent. We use regex to extract the 4-digit year across multiple date columns, coalescing them into an `ultimate_year`. Any movie lacking a verifiable year is dropped.
2. **CPI Adjustment (2024 Base Year):** We utilize historical US Consumer Price Index (CPI) data to convert every historical `ultimate_budget` and `ultimate_revenue` into **Real 2024 Dollars** (`real_budget`, `real_revenue`).
3. **Formulating the North Star:** With the currency standardized, we calculate the absolute `real_profit` and define our primary binary classification target: `is_profitable` (1 if Real Profit > 0, else 0).

In [ ]:
# --- PHASE II: FINANCIAL STANDARDIZATION & MACROECONOMICS ---
import numpy as np
import pandas as pd

print(f"Starting Phase II with {len(enriched_clean):,} financially verified movies...")
df_finance = enriched_clean.copy()

# ==========================================
# 1. Establish Temporal SSOT (Release Year)
# ==========================================
# Extract the 4-digit year from all potential date columns using Regex
y_base = df_finance.get('year', pd.Series(np.nan, index=df_finance.index)).astype(str).str.extract(r'(\d{4})')[0]
date_tmdb = df_finance.get('release_date', pd.Series(np.nan, index=df_finance.index)).astype(str).str.extract(r'(\d{4})')[0]
date_base = df_finance.get('released', pd.Series(np.nan, index=df_finance.index)).astype(str).str.extract(r'(\d{4})')[0]

# Coalesce to create the ultimate source of truth for the release year
df_finance['ultimate_year'] = y_base.combine_first(date_tmdb).combine_first(date_base)

# Drop records lacking a verifiable release year, as inflation cannot be calculated without it
df_finance = df_finance.dropna(subset=['ultimate_year'])
df_finance['ultimate_year'] = df_finance['ultimate_year'].astype(int)

# ==========================================
# 2. Inflation Adjustment (Base Year: 2024)
# ==========================================
# US Consumer Price Index (CPI) Data
cpi_data = {
    1980: 82.4,  1981: 90.9,  1982: 96.5,  1983: 99.6,  1984: 103.9,
    1985: 107.6, 1986: 109.6, 1987: 113.6, 1988: 118.3, 1989: 124.0,
    1990: 130.7, 1991: 136.2, 1992: 140.3, 1993: 144.5, 1994: 148.2,
    1995: 152.4, 1996: 156.9, 1997: 160.5, 1998: 163.0, 1999: 166.6,
    2000: 172.2, 2001: 177.1, 2002: 179.9, 2003: 184.0, 2004: 188.9,
    2005: 195.3, 2006: 201.6, 2007: 207.3, 2008: 215.3, 2009: 214.5,
    2010: 218.1, 2011: 224.9, 2012: 229.6, 2013: 233.0, 2014: 236.7,
    2015: 237.0, 2016: 240.0, 2017: 245.1, 2018: 251.1, 2019: 255.7,
    2020: 258.8, 2021: 271.0, 2022: 292.7, 2023: 304.7, 2024: 313.7
}
CPI_2024 = 313.7

def adjust_for_inflation(row, col_name):
    """Adjusts historical dollar amounts to 2024 equivalents using CPI data."""
    year = row['ultimate_year']
    amount = row[col_name]
    if year not in cpi_data:
        return amount  # Return unadjusted amount if year falls outside CPI dictionary
    return amount * (CPI_2024 / cpi_data[year])

# Apply the inflation adjustment function to the primary financial metrics
df_finance['real_budget'] = df_finance.apply(lambda x: adjust_for_inflation(x, 'ultimate_budget'), axis=1)
df_finance['real_revenue'] = df_finance.apply(lambda x: adjust_for_inflation(x, 'ultimate_revenue'), axis=1)

# ==========================================
# 3. Target Formulation
# ==========================================
# Calculate absolute profit and the primary binary classification target
df_finance['real_profit'] = df_finance['real_revenue'] - df_finance['real_budget']
df_finance['is_profitable'] = (df_finance['real_profit'] > 0).astype(int)

print(f"Phase II Complete! Financial targets formulated for {len(df_finance):,} movies.")

# Diagnostic check
quick_column_summary(df_finance[['ultimate_year', 'real_budget', 'real_revenue', 'real_profit', 'is_profitable']], 'Inflation Adjusted Financials')

Starting Phase II with 11,668 financially verified movies...
Phase II Complete! Financial targets formulated for 5,705 movies.

📋 Column Summary for `Inflation Adjusted Financials`



,Column,Data Type,NA Count,% Missing
0,ultimate_year,int64,0,0.0
1,real_budget,float64,0,0.0
2,real_revenue,float64,0,0.0
3,real_profit,float64,0,0.0
4,is_profitable,int64,0,0.0


# Phase III: Feature Engineering (The R.I.C.E. Framework + API Metadata Rescue)

With a mathematically sound financial target (`real_profit`) established, we must translate our raw categorical data into predictive business signals and impute missing values to maximize our dataset's footprint.

### Part 1: The R.I.C.E. Framework
We engineer strategic features based on industry mechanics:
1. **IP & Franchise Power (`is_sequel`):** Franchises represent established market reach. We flag sequels to account for lower customer acquisition costs.
2. **Scale (`is_major_studio`):** We isolate the "Big 6" studios to proxy the massive distribution pipelines and marketing budgets unavailable to independent films.
3. **Release Strategy (`is_blockbuster_season`):** We parse release dates to identify strategic placement (e.g., Summer/Holiday), which dictates box office ceilings.

### Part 2: API Metadata Rescue
Raw datasets frequently suffer from missing values in critical columns like `score`, `vote_count`, and `rating`. Rather than dropping these rows and losing valuable financial data, we deploy a programmatic pipeline to query the **TMDB API**.
* We parse existing textual identifiers to fetch the exact database matches.
* We impute missing age ratings, production countries, and audience reception metrics.
* We explicitly convert false zeros (e.g., an unrated movie listed as a `0.0` score) into `NaN` to prevent the algorithm from misinterpreting a lack of data as a negative quality signal.

## Feature Engineering (Part 1: R.I.C.E Signals)

In [ ]:
# --- PHASE III: FEATURE ENGINEERING (Part 1: R.I.C.E. Signals) ---
import pandas as pd
import numpy as np

print(f"Starting Phase III with {len(df_finance):,} movies...")
df_features = df_finance.copy()

# ==========================================
# 0. The Pre-Emptive Purge
# ==========================================
# Remove missing runtimes
df_features = df_features.dropna(subset=['runtime'])

# Convert the literal string "nan" to an actual missing value, then drop it
df_features['movie_id'] = df_features['movie_id'].replace('nan', np.nan)
df_features = df_features.dropna(subset=['movie_id'])

print(f"Valid movies after dropping missing IDs and Runtimes: {len(df_features):,}")

# ==========================================
# 1. IP / Franchise Power (Reach Signal)
# ==========================================
sequel_keywords = ['sequel', 'trilogy', 'prequel', 'spin off', 'remake', 'franchise', 'shared universe']

def check_ip_status(row):
    """Identifies if a movie is part of an existing intellectual property/franchise."""
    kw = str(row.get('keywords', '')).lower()
    for word in sequel_keywords:
        if word in kw:
            return 1

    title = str(row.get('name', '')).lower()
    # Check for common sequel naming conventions (e.g., "Shrek 2", "Kill Bill: Vol. 1")
    if any(suffix in title for suffix in [' 2', ' 3', ' part ', ' vol']):
        return 1
    return 0

df_features['is_sequel'] = df_features.apply(check_ip_status, axis=1)

# ==========================================
# 2. Historical Confidence (Target Encoding)
# ==========================================
def calculate_win_rates(df, col_name):
    """
    Calculates the historical profitability win rate for categorical features.
    Missing values are imputed with the global average win rate to neutralize risk.
    """
    win_rates = df.groupby(col_name)['is_profitable'].mean()
    global_win_rate = df['is_profitable'].mean()
    return df[col_name].map(win_rates).fillna(global_win_rate)

df_features['director_win_rate'] = calculate_win_rates(df_features, 'director')
df_features['star_win_rate'] = calculate_win_rates(df_features, 'star')
df_features['writer_win_rate'] = calculate_win_rates(df_features, 'writer')
df_features['production_win_rate'] = calculate_win_rates(df_features, 'company')
df_features['rating_win_rate'] = calculate_win_rates(df_features, 'rating')

# ==========================================
# 3. Studio Scale (Effort Signal)
# ==========================================
major_studios = [
    'Warner Bros.', 'Universal Pictures', 'Columbia Pictures',
    'Paramount Pictures', 'Twentieth Century Fox', 'Walt Disney Pictures',
    'New Line Cinema'
]

pattern = '|'.join(major_studios)
df_features['is_major_studio'] = np.where(
    df_features['company'].astype(str).str.contains(pattern, case=False, na=False), 1, 0
)

# ==========================================
# 4. Release Strategy (Timing Signals)
# ==========================================
# Parse release dates safely to extract seasonality and weekday data
df_features['parsed_date'] = pd.to_datetime(df_features.get('release_date', ''), errors='coerce')

# A. Blockbuster Season (Summer: May-Jul | Holidays: Nov-Dec)
df_features['is_blockbuster_season'] = np.where(
    df_features['parsed_date'].dt.month.isin([5, 6, 7, 11, 12]), 1, 0
)

# B. Weekend Release (Thursday previews = 3, Friday = 4, Saturday = 5)
df_features['is_weekend_release'] = np.where(
    df_features['parsed_date'].dt.dayofweek.isin([3, 4, 5]), 1, 0
)

# Clean up temporary parsing column
df_features = df_features.drop(columns=['parsed_date'])

print("✅ Phase III (Part 1) Complete: Confidence, Scale, and Timing Signals Engineered.")

Starting Phase III with 5,705 movies...
Valid movies after dropping missing IDs and Runtimes: 5,350
✅ Phase III (Part 1) Complete: Confidence, Scale, and Timing Signals Engineered.


In [ ]:
# --- PHASE 3.2: API GENRE IMPUTATION ---
import requests
import time
import pandas as pd
import numpy as np
from IPython.display import clear_output

df_genres = df_features.copy()

# 1. INSERT YOUR TMDB API KEY HERE (Use the 'API Key (v3 auth)', NOT the Read Access Token)
TMDB_API_KEY = '7f02ffe39e776284f852e615642c96ca'

# ==========================================
# DIAGNOSTIC: Test the API Key First
# ==========================================
print("Testing TMDB API Connection...")
test_url = f"https://api.themoviedb.org/3/search/movie?api_key={TMDB_API_KEY}&query=Avatar"
test_response = requests.get(test_url).json()

if 'status_code' in test_response and test_response['status_code'] != 1:
    print(f"🚨 API ERROR: TMDB rejected the key. Message: {test_response.get('status_message')}")
    raise ValueError("Stop: Please fix your API key before continuing.")
else:
    print("✅ API Connection Successful! Proceeding with rescue...\n")
    time.sleep(1)

# ==========================================
# STEP 1: Extract ONE Primary Genre from raw text
# ==========================================
major_genres = ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary',
                'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery',
                'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War', 'Western']

# Make sure this matches your raw genre column name before the one-hot encoding!
raw_genre_col = 'genres'

def get_primary_genre(text):
    if pd.isna(text): return None
    text_lower = str(text).lower()
    for genre in major_genres:
        search_term = 'sci-fi' if genre == 'Science Fiction' else genre.lower()
        if search_term in text_lower:
            return genre
    return None

df_genres['primary_genre'] = df_genres[raw_genre_col].apply(get_primary_genre)
initial_unclassified = df_genres['primary_genre'].isna().sum()
print(f"⚠️ Initial Scan: {initial_unclassified} movies are completely unclassified.")

# ==========================================
# STEP 2: The API Rescue Mission
# ==========================================
# Using 'movie_id' as the title column based on your CSV
movies_to_fix = df_genres[df_genres['primary_genre'].isna()]['movie_id'].tolist()

tmdb_mapping = {
    28: 'Action', 12: 'Adventure', 16: 'Animation', 35: 'Comedy', 80: 'Crime',
    99: 'Documentary', 18: 'Drama', 10751: 'Family', 14: 'Fantasy', 36: 'History',
    27: 'Horror', 10402: 'Music', 9648: 'Mystery', 10749: 'Romance', 878: 'Science Fiction',
    10770: 'TV Movie', 53: 'Thriller', 10752: 'War', 37: 'Western'
}

rescued_count = 0
rescued_genres_tracker = []

for i, title_val in enumerate(movies_to_fix):
    if i % 50 == 0 and i > 0:
        print(f"API Fetching: {i} / {len(movies_to_fix)} movies processed...")

    try:
        url = f"https://api.themoviedb.org/3/search/movie?api_key={TMDB_API_KEY}&query={title_val}"
        response = requests.get(url).json()

        if response.get('results') and len(response['results']) > 0:
            top_result = response['results'][0]
            genre_ids = top_result.get('genre_ids', [])

            for gid in genre_ids:
                if gid in tmdb_mapping:
                    rescued_genre = tmdb_mapping[gid]
                    df_genres.loc[df_genres['movie_id'] == title_val, 'primary_genre'] = rescued_genre
                    rescued_count += 1
                    rescued_genres_tracker.append(rescued_genre)
                    break
    except Exception as e:
        print(f"Error on {title_val}: {e}") # Now it will actually tell us if something breaks

    time.sleep(0.05)

# ==========================================
# STEP 3: One-Hot Encoding & Final Audit
# ==========================================
genre_dummies = pd.get_dummies(df_genres['primary_genre'], dtype=int)
df_genres = df_genres.drop(columns=[g for g in major_genres if g in df_genres.columns], errors='ignore')
df_genres = pd.concat([df_genres, genre_dummies], axis=1)

for g in major_genres:
    if g not in df_genres.columns:
        df_genres[g] = 0

final_unclassified = df_genres['primary_genre'].isna().sum()

# ==========================================
# STEP 4: Print the Fix Report & Drop Invalid Rows
# ==========================================
clear_output()
print("✅ GENRE RESCUE COMPLETE\n" + "="*30)
print(f"📉 Unclassified Before: {initial_unclassified}")
print(f"🛠️ Movies Successfully Fixed via API: {rescued_count}")
print(f"🚨 Remaining Unclassified: {final_unclassified} (These will be dropped)\n")

if rescued_count > 0:
    print("📊 Distribution of the Rescued Genres:")
    print(pd.Series(rescued_genres_tracker).value_counts().to_string())

df_genres = df_genres.dropna(subset=['primary_genre']).copy()
df_genres = df_genres.drop(columns=['primary_genre'])
print(f"\nFinal Dataset Size: {len(df_genres)} mutually-exclusive, single-genre movies ready for Phase 4.")

✅ GENRE RESCUE COMPLETE
📉 Unclassified Before: 4205
🛠️ Movies Successfully Fixed via API: 4184
🚨 Remaining Unclassified: 21 (These will be dropped)

📊 Distribution of the Rescued Genres:
Comedy             1018
Drama               943
Action              551
Horror              317
Adventure           266
Crime               217
Thriller            163
Science Fiction     114
Romance             109
Fantasy             108
Family              104
Animation            98
Mystery              58
Documentary          27
Music                27
Western              23
War                  23
History              15
TV Movie              3

Final Dataset Size: 5329 mutually-exclusive, single-genre movies ready for Phase 4.


# Phase III (Continued Feature Engineering): NLP (Thematic Reach)

While categorical features like Genre provide a high-level understanding of a movie, they miss the nuanced hooks that actually draw audiences to theaters. To translate unstructured text into a mathematical format, we employ Natural Language Processing (NLP).

1. **Semantic Consolidation:** We merge the movie's plot summary (`overview`) with its community-tagged `keywords`. Before merging, we apply an Abstract Syntax Tree (AST) parser to clean any raw JSON formatting out of the keyword data.
2. **Thematic Extraction (`CountVectorizer`):** We deploy a text vectorizer to analyze the entire corpus. By utilizing a custom "stop-words" dictionary, we filter out standard English (e.g., "the", "and") as well as generic meta-terms (e.g., "film", "movie", "based").
3. **Dimensionality Control:** We strictly cap the extraction at `max_features=30`. This prevents the "Curse of Dimensionality" (where too many columns crash the model) and forces the algorithm to identify only the most dominant, predictive narrative themes across Hollywood (e.g., `theme_murder`, `theme_alien`, `theme_wedding`).

**The Business Value:** These NLP features act as highly specific **Reach Signals**, allowing the model to correlate specific plot elements directly with box office profitability.

In [ ]:
# --- PHASE 3.3: METADATA & KEYWORD IMPUTATION (TMDB API) ---
import requests
import time
import pandas as pd
import numpy as np
import ast
from IPython.display import clear_output

TMDB_API_KEY = '7f02ffe39e776284f852e615642c96ca'
print("Initiating Unified API Imputation...")

# PIPELINE HANDOFF: Pulls from Phase 3.2
df_api = df_genres.copy()

# 1. Initialize columns
cols_to_init = ['rating', 'country', 'vote_count', 'score', 'keywords']
for col in cols_to_init:
    if col not in df_api.columns:
        df_api[col] = np.nan

# 2. Clean invalid zeros
df_api['score'] = pd.to_numeric(df_api['score'], errors='coerce').replace(0, np.nan)
df_api['vote_count'] = pd.to_numeric(df_api['vote_count'], errors='coerce').replace(0, np.nan)

# 3. Safely parse any existing JSON keywords BEFORE the API fetch
def parse_keyword_json(text):
    if pd.isna(text) or text == '': return np.nan
    if isinstance(text, str) and text.strip().startswith('['):
        try:
            parsed_list = ast.literal_eval(text)
            words = [item['name'] for item in parsed_list if 'name' in item]
            return " ".join(words) if words else np.nan
        except: return np.nan
    return text

df_api['keywords'] = df_api['keywords'].apply(parse_keyword_json)

# 4. API Fetch (Using 'title' because 'movie_id' contains corrupted string data)
title_col = 'title'
missing_mask = df_api[cols_to_init].isna().any(axis=1)
valid_title_mask = df_api[title_col].notna()
movies_to_fix = df_api[missing_mask & valid_title_mask][title_col].astype(str).unique().tolist()

for i, title_val in enumerate(movies_to_fix):
    if i % 50 == 0 and i > 0:
        print(f"API Fetching: {i} / {len(movies_to_fix)} records processed...")

    try:
        search_url = f"https://api.themoviedb.org/3/search/movie?api_key={TMDB_API_KEY}&query={title_val}"
        search_resp = requests.get(search_url).json()

        if search_resp.get('results') and len(search_resp['results']) > 0:
            movie_id = search_resp['results'][0]['id']
            url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={TMDB_API_KEY}&append_to_response=release_dates,keywords"
            resp = requests.get(url).json()

            if 'id' in resp:
                mask = df_api[title_col] == title_val

                if pd.isna(df_api.loc[mask, 'vote_count'].iloc[0]) and resp.get('vote_count', 0) > 0:
                    df_api.loc[mask, 'vote_count'] = resp.get('vote_count')
                if pd.isna(df_api.loc[mask, 'score'].iloc[0]) and resp.get('vote_average', 0) > 0:
                    df_api.loc[mask, 'score'] = resp.get('vote_average')

                if pd.isna(df_api.loc[mask, 'country'].iloc[0]):
                    countries = resp.get('production_countries', [])
                    if countries:
                        df_api.loc[mask, 'country'] = countries[0].get('iso_3166_1')

                if pd.isna(df_api.loc[mask, 'keywords'].iloc[0]):
                    kws = resp.get('keywords', {}).get('keywords', [])
                    if kws:
                        df_api.loc[mask, 'keywords'] = " ".join([k['name'] for k in kws])

                if pd.isna(df_api.loc[mask, 'rating'].iloc[0]):
                    for country_data in resp.get('release_dates', {}).get('results', []):
                        if country_data['iso_3166_1'] == 'US':
                            for release in country_data.get('release_dates', []):
                                cert = release.get('certification')
                                if cert:
                                    df_api.loc[mask, 'rating'] = cert
                                    break
                            break
    except Exception:
        pass
    time.sleep(0.08)

rating_map = {'X': 'NC-17', 'Approved': 'PG', 'Passed': 'PG', 'Not Rated': 'Unrated', 'TV-MA': 'NC-17'}
df_api['rating'] = df_api['rating'].replace(rating_map)

clear_output()
print("Phase 3.3 Complete: Data Imputation via TMDB API.")

Phase 3.3 Complete: Data Imputation via TMDB API.


In [ ]:
# --- PHASE 3.4: NLP THEMATIC FEATURE ENGINEERING ---
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
import pandas as pd

print("Applying NLP thematic feature extraction...")

# PIPELINE HANDOFF: Pulls from Phase 3.3
df_nlp = df_api.copy()

df_nlp['combined_text'] = df_nlp['overview'].fillna('') + ' ' + df_nlp['keywords'].fillna('')

custom_stops = [
    'film', 'movie', 'based', 'novel', 'story', 'time', 'new', 'york',
    'duringcreditsstinger', 'id', 'character', 'life', 'man', 'woman',
    'young', 'world', 'way', 'make', 'just', 'does', 'like'
]
all_stop_words = list(ENGLISH_STOP_WORDS) + custom_stops

vectorizer = CountVectorizer(stop_words=all_stop_words, max_features=30, token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b')
word_matrix = vectorizer.fit_transform(df_nlp['combined_text'])

nlp_words = vectorizer.get_feature_names_out()
nlp_columns = [f"theme_{word}" for word in nlp_words]

df_nlp_features = pd.DataFrame(word_matrix.toarray(), columns=nlp_columns, index=df_nlp.index)
df_nlp = pd.concat([df_nlp.drop(columns=['combined_text']), df_nlp_features], axis=1)

print(f"Phase 3.4 Complete: Engineered {len(nlp_columns)} NLP features.")

Applying NLP thematic feature extraction...
Phase 3.4 Complete: Engineered 30 NLP features.


# Phase IV: Final Formatting & Data Leakage Prevention

The final phase of our preprocessing pipeline ensures the dataset is mathematically clean, dense, and strictly protects the integrity of the future Machine Learning model.

1. **Target Vectorization:** We finalize our binary classification target (`is_profitable`) using highly efficient NumPy vectorization based on inflation-adjusted revenue and budget.
2. **The Data Leakage Barrier (Strict Whitelisting):** The most critical step in predictive modeling is ensuring the algorithm cannot "cheat" by looking at unstructured or redundant data. We enforce a strict column whitelist:
    * **Retained:** Core financials, R.I.C.E. strategy metrics, talent identifiers, Genres, and our 30 NLP Themes.
    * **Purged (Redundancy & Noise):** We permanently drop unstructured strings (`title`, `overview`), unstable date formats (`release_date`, replaced by year/month integers), and redundant targets (`worldwide_gross`, replaced by our inflation-adjusted metrics).
3. **The Final Purge:** Any row still missing core financial data (`real_budget`, `real_revenue`) after all imputation attempts is dropped.

The resulting matrix is a pristine, mathematically dense dataset ready for Exploratory Data Analysis (EDA) and Machine Learning deployment.

In [ ]:
# --- PHASE IV: TARGET FORMULATION, LEAKAGE PREVENTION & FEATURE SELECTION ---
import pandas as pd
import numpy as np

print("Initiating Target Formulation and Feature Selection...")

# PIPELINE HANDOFF: Pulls from Phase 3.4
df_final = df_nlp.copy()

# 1. Target Formulation & Mathematical Enrichment
if 'real_profit' not in df_final.columns:
    df_final['real_profit'] = df_final['real_revenue'] - df_final['real_budget']
if 'is_profitable' not in df_final.columns:
    df_final['is_profitable'] = np.where(df_final['real_profit'] > 0, 1, 0)

# Restore ROI and log_real_revenue directly into the preprocessing pipeline
if 'ROI' not in df_final.columns:
    # Safely calculate ROI, avoiding division by zero
    df_final['ROI'] = np.where(df_final['real_budget'] > 0,
                               df_final['real_profit'] / df_final['real_budget'],
                               np.nan)

if 'log_real_revenue' not in df_final.columns:
    # np.log1p handles log(1+x) to prevent errors on absolute zeros
    df_final['log_real_revenue'] = np.log1p(df_final['real_revenue'])

# 2. The Comprehensive Whitelist Definition
core_cols = ['movie_id', 'real_budget', 'real_revenue', 'real_profit', 'is_profitable', 'ROI', 'log_real_revenue']

# Restoring 'year', 'release_year', and 'is_weekend_release'
potential_strategy = ['is_sequel', 'is_major_studio', 'is_blockbuster_season', 'is_weekend_release', 'ultimate_year', 'released','ultimate_year', 'runtime']
strategy_cols = [c for c in potential_strategy if c in df_final.columns]

# Restoring ALL win_rates while keeping the raw string names
potential_talent = [
    'director', 'star', 'writer', 'producer', 'production_company',
    'director_win_rate', 'star_win_rate', 'writer_win_rate',
    'production_win_rate', 'rating_win_rate'
]
talent_cols = [c for c in potential_talent if c in df_final.columns]

eda_cols = ['rating', 'country', 'vote_count', 'score']

genre_cols = ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary',
              'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery',
              'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War', 'Western']

nlp_cols = [c for c in df_final.columns if str(c).startswith('theme_')]

# Combine everything
whitelist = core_cols + strategy_cols + talent_cols + eda_cols + genre_cols + nlp_cols

# Apply Whitelist and drop unrecoverable financials
final_columns = [c for c in whitelist if c in df_final.columns]
df_final = df_final[final_columns].copy()
df_final = df_final.dropna(subset=['real_budget', 'real_revenue'])
df_final = df_final.rename(columns={'ultimate_year': 'released_year', 'released': 'released_date'})
df_final = df_final.loc[:, ~df_final.columns.duplicated()]

print(f"Phase 4.1 Complete. Final Dataset Shape: {df_final.shape[0]} Rows, {df_final.shape[1]} Features.")

Initiating Target Formulation and Feature Selection...
Phase 4.1 Complete. Final Dataset Shape: 5329 Rows, 75 Features.


In [ ]:
# Check for data types and NA's
quick_column_summary(df_final, 'df_final')


📋 Column Summary for `df_final`



,Column,Data Type,NA Count,% Missing
0,score,float64,2751,51.62
1,vote_count,float64,2372,44.51
2,rating,object,7,0.13
3,country,object,1,0.02
4,is_profitable,int64,0,0.00
...,...,...,...,...
70,theme_town,int64,0,0.00
71,theme_true,int64,0,0.00
72,theme_war,int64,0,0.00
73,theme_wife,int64,0,0.00


In [ ]:
# Check how it looks
df_final.head()

,movie_id,real_budget,real_revenue,real_profit,is_profitable,ROI,log_real_revenue,is_sequel,is_major_studio,is_blockbuster_season,...,theme_police,theme_relationship,theme_school,theme_secret,theme_son,theme_town,theme_true,theme_war,theme_wife,theme_years
5,*batteries not included,6.903609e+07,1.797390e+08,1.107030e+08,1,1.603552,19.007017,0,1,0,...,0,0,0,0,0,0,0,0,0,0
10,10 cloverfield lane,6.535417e+06,1.415394e+08,1.350040e+08,1,20.657284,18.768089,0,1,0,...,0,0,0,0,0,0,0,0,0,0
13,10 things i hate about you,5.648860e+07,1.006977e+08,4.420907e+07,1,0.782619,18.427633,0,0,0,...,0,0,1,0,0,1,0,0,0,0
14,10 to midnight,1.423618e+07,2.260023e+07,8.364048e+06,1,0.587520,16.933471,0,0,0,...,0,0,0,0,0,0,0,0,0,0
15,"10,000 bc",1.529889e+08,3.930855e+08,2.400966e+08,1,1.569373,19.789538,0,1,0,...,0,0,0,0,0,0,0,0,0,0


# Export Final Dataset

In [ ]:
# --- DATA EXPORT ---
# Run this cell to download the finalized dataset for EDA and Modeling.

file_name = 'greenlight_model_data.csv'
df_final.to_csv(file_name, index=False)

from google.colab import files
try:
    files.download(file_name)
    print(f"✅ Successfully triggered download for '{file_name}'.")
except Exception as e:
    print(f"✅ Saved '{file_name}' to local Colab session storage.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Successfully triggered download for 'greenlight_model_data.csv'.


## 📚 Appendix A: The Final Data Dictionary

This table details the exact composition of our final exported dataset (`greenlight_model_data_FINAL.csv`). It maps each feature to its origin, its data type, and its designated analytical category.

| Column Name(s) | Signal Type | Feature Origin | Data Type | Source Dataset(s) |
| :--- | :--- | :--- | :--- | :--- |
| **`movie_id`** | Primary Key | Engineered (Normalized) | String/Int | *All Datasets* |
| **`is_profitable`** | Target (Classification) | Engineered (Math) | Binary (0/1) | *Phase IV Target Formulation* |
| **`real_profit`, `real_revenue`, `ROI`, `log_real_revenue`** | Targets (Regression/EDA) | Engineered (Math & CPI) | Continuous | *Phase II & IV Engineering* |
| **`real_budget`** | Effort (Capital) | Engineered (CPI Adjusted) | Continuous | `financial_data`, `movie_franchises` |
| **`year`, `release_year`, `release_month`** | Effort (Context) | Raw / Parsed | Integer | `movie_franchises`, `tmdb_data` |
| **`runtime`** | Effort (Scale) | Raw / Coalesced | Continuous | `movie_franchises`, `tmdb_data` |
| **`is_major_studio`** | Effort (Marketing) | Engineered (Regex) | Binary (0/1) | *Phase III Engineering* |
| **`director`, `star`, `writer`, `producer`, `production_company`** | Confidence (Raw) | Raw / Coalesced | String (Categorical)| `meta_data`, `tmdb_data` |
| **`director_win_rate`, `star_win_rate`, `writer_win_rate`, `production_win_rate`, `rating_win_rate`** | Confidence (Encoded)| Engineered (Target Encoding) | Continuous (0.0-1.0)| *Phase III Engineering* |
| **`is_sequel`** | Reach (IP) | Engineered (Heuristics) | Binary (0/1) | *Phase III Engineering* |
| **`is_blockbuster_season`, `is_weekend_release`** | Reach (Timing) | Engineered (Date Parsing) | Binary (0/1) | `tmdb_data` (`release_date`) |
| **`rating`, `country`** | Reach (Constraints) | API Imputed | String (Categorical)| `tmdb_api` |
| **`score`, `vote_count`** | Reach (EDA Only)* | API Imputed | Continuous | `tmdb_api` |
| **`Action` through `Western`** *(19 Columns)* | Reach (Demographics) | Raw / Imputed | Binary (0/1) | `movie_franchises`, `meta_data` |
| **`theme_agent` through `theme_years`** *(30 Columns)* | Reach (NLP Content) | Engineered (CountVectorizer)| Binary/Count | `meta_data`, `tmdb_api` (`keywords`) |

*\*Note: `score` and `vote_count` represent post-release reality. They are retained strictly for Exploratory Data Analysis (EDA) and must be dropped prior to predictive model training to prevent Data Leakage.*

---

## 🎯 Appendix B: The R.I.C.E. Feature Architecture

To ensure our Machine Learning model behaves like a Hollywood Studio Executive, our features are strictly organized into the **R.I.C.E. Framework**. This architecture dictates *why* the algorithm evaluates specific data points to generate its Greenlight probability.

### 📡 1. REACH (Market Size & Audience Appeal)
*These features dictate how wide the potential consumer base is for the film.*
* **Franchise Power (`is_sequel`)**: Built-in IP audiences significantly lower customer acquisition costs.
* **Release Timing (`is_blockbuster_season`, `is_weekend_release`)**: Strategic placement in high-traffic theater windows.
* **Market Constraints (`rating`, `country`)**: MPAA age ratings and production countries dictate the Total Addressable Market.
* **Demographics (19 Genre Columns)**: Maps the film to specific audience preferences.
* **Thematic Hooks (30 NLP Columns)**: Mathematically identifies the core narrative elements that draw audiences in.
* **Post-Release Hype (`score`, `vote_count`)**: Used retroactively during EDA to analyze audience engagement and quality metrics.

### 💥 2. IMPACT (Financial Outcomes)
*These are our "North Star" Targets. The pipeline is optimized to predict these specific outcomes.*
* **Binary Target (`is_profitable`)**: The primary classification goal (1 = Profit > 0).
* **Continuous Targets (`real_profit`, `real_revenue`, `ROI`, `log_real_revenue`)**: Metrics tracking the exact magnitude of the financial impact and return on investment (adjusted for 2024 inflation).

### 🤝 3. CONFIDENCE (Execution Reliability)
*These features assess the track record of the human capital responsible for delivering the product.*
* **The Talent Base (`director`, `star`, `writer`, `producer`, `production_company`)**: Raw categorical identifiers for the primary creative team.
* **Target Encodings (`_win_rate` columns)**: The historical probability of the attached personnel delivering a profitable film, converting raw names into mathematical risk profiles.

### 💰 4. EFFORT (Capital & Scale Risk)
*These features measure how difficult, expensive, and resource-intensive the product is to bring to market.*
* **`real_budget`**: The literal capital at risk, adjusted for 2024 inflation.
* **`is_major_studio`**: A proxy for "Marketing Effort." A Big 6 studio guarantees massive distribution pipelines.
* **`runtime`**: A proxy for physical production scope. Longer films require more shooting days and extended post-production.
* **Temporal Context (`year`, `release_year`, `release_month`)**: Contextual effort baselines to track industry paradigm shifts over time.

---

## 🧠 Appendix C: Methodology & Key Data Decisions

To build a production-grade predictive model, data integrity must take precedence over data volume. Below are the core methodological decisions made during the data engineering pipeline.

### 1. The "Financial North Star" Purge (No Financial Imputation)
* **The Decision:** Any movie missing a verifiable Budget or Revenue across all coalesced datasets was strictly dropped.
* **The Justification:** A financial Greenlight model relies entirely on ground truth. Imputing a missing budget using global medians introduces fatal bias and target leakage. We sacrificed significant raw data volume to guarantee absolute financial integrity for the remaining theatrical releases.

### 2. Target Encoding & Categorical Imputation
* **The Decision:** Engineered "Win Rates" for personnel using historical profitability. For movies missing specific crew metadata, missing values were imputed with the global baseline average win rate.
* **The Justification:** Dropping a validated $100M budget/revenue row because the web scraper missed the screenwriter is highly inefficient. Assigning the "Global Average" to an unknown crew member mathematically neutralizes their risk impact, allowing the model to learn from the rest of the valid features without skewing the distribution.

### 3. Programmatic API Imputation (TMDB)
* **The Decision:** Rather than dropping rows due to missing metadata, we deployed a unified pipeline to query the TMDB API to impute `score`, `vote_count`, `rating`, `country`, and missing `keywords`.
* **The Justification:** If a row possesses perfect Budget and Revenue data, dropping it because it lacks a "PG-13" tag destroys highly valuable target data. The API imputation salvaged thousands of data points, expressly converting "false zeros" (unrated movies stored as a 0.0 score) into `NaN` prior to the API ping to prevent negative quality signal errors.

### 4. Logarithmic & Ratio Transformations (`log_real_revenue`, `ROI`)
* **The Decision:** Alongside the binary classification target, `log_real_revenue` and `ROI` were engineered and explicitly included in the final export.
* **The Justification:** Theatrical box office returns follow a massive power-law (fat-tail) distribution. Log transformation (`np.log1p`) standardizes the variance, allowing the algorithm to learn the underlying mechanics of a successful movie rather than just memorizing extreme outliers. Pre-calculating this in the preprocessing pipeline ensures the data is perfectly locked and ready for immediate EDA.

### 5. Macroeconomic Standardization (CPI Indexing)
* **The Decision:** All financial metrics (Budget, Revenue, Profit) were adjusted to 2024 USD equivalents using the US Consumer Price Index.
* **The Justification:** A `$50` million box office return in 1985 represents an entirely different scale of cultural impact than a `$50` million return in 2024. Standardizing the currency prevents the model from inherently penalizing older films.